# AtlasOps bug lab — Qwen2.5-7B on a Kaggle T4

Serves `qwen2.5:7b-instruct` from Ollama behind a key-checking proxy, then exposes
it through an ngrok HTTPS URL that the bug lab container on the Mac calls.

Implements section 6 of *AtlasOps Bug Lab: Run Plan* (27 October 2026).

## Before running

- Kaggle account with a verified phone number (required for GPU and internet).
- Notebook settings: **Accelerator GPU T4**, **Internet on**.
- Two notebook secrets under Add-ons -> Secrets:
  - `NGROK_AUTHTOKEN` — from your ngrok account.
  - `BUG_LAB_LLM_API_KEY` — generate with `openssl rand -hex 32`.

## Order matters

Cells run in order: install, start and warm the model, start the proxy, then open the
tunnel **last**. An earlier run in this repository hit HTTP 524 opening a tunnel while
the model was still loading (`docs/project/G4_V38_TRANSPORT_DIAGNOSTICS_V1.md`).
Warming the model on the notebook first is what avoids that.

## Limits

- GPU session ends after at most 12 hours; the URL changes every session.
- Weekly GPU quota is commonly about 30 hours — check your account settings.
- Read Kaggle's terms before relying on a tunnel. Never share the URL publicly.

The printed `BUG_LAB_LLM_BASE_URL` goes in the Mac's env file. The key never is printed.


In [ ]:
# Cell 1 — secrets and installs. Newer Ollama install scripts need zstd.
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
NGROK_TOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
API_KEY = secrets.get_secret("BUG_LAB_LLM_API_KEY")

!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pyngrok fastapi uvicorn httpx


In [ ]:
# Cell 2 — start Ollama, pull the model, warm it up on the notebook itself.
# Ollama's default context window can cut off long prompts, so raise it.
import os, subprocess, time
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434",
           OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_KEEP_ALIVE="-1")
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen2.5:7b-instruct
!curl -s http://127.0.0.1:11434/v1/chat/completions -H "Content-Type: application/json" \
  -d '{"model":"qwen2.5:7b-instruct","messages":[{"role":"user","content":"Reply with OK"}],"max_tokens":5}'
!nvidia-smi --query-gpu=name,memory.used --format=csv


In [ ]:
# Cell 3 — the key-checking proxy. Ollama has no authentication of its own.
# The key comes from an environment variable, so it is never written to /kaggle/working.
PROXY = """
import hmac, os, httpx
from fastapi import FastAPI, Request, Response
KEY = os.environ["PROXY_KEY"]
upstream = httpx.AsyncClient(base_url="http://127.0.0.1:11434", timeout=600)
app = FastAPI()

@app.api_route("/v1/{path:path}", methods=["GET", "POST"])
async def forward(path: str, request: Request):
    sent = request.headers.get("authorization", "")
    if not hmac.compare_digest(sent.encode(), f"Bearer {KEY}".encode()):
        return Response(status_code=401)
    r = await upstream.request(request.method, f"/v1/{path}", content=await request.body(),
                               headers={"content-type": "application/json"})
    return Response(r.content, status_code=r.status_code,
                    media_type=r.headers.get("content-type"))
"""
open("/kaggle/working/proxy.py", "w").write(PROXY)
subprocess.Popen(["python", "-m", "uvicorn", "proxy:app", "--host", "127.0.0.1", "--port", "8000"],
                 cwd="/kaggle/working", env=dict(os.environ, PROXY_KEY=API_KEY),
                 stdout=open("/kaggle/working/proxy.log", "w"), stderr=subprocess.STDOUT)
time.sleep(3)
!curl -s -o /dev/null -w "without key: %{http_code}\n" http://127.0.0.1:8000/v1/models   # expect 401


In [ ]:
# Cell 4 — open the tunnel last, once the model has answered on the notebook itself.
# This prints the URL, not the key.
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(8000, "http")
print("BUG_LAB_LLM_BASE_URL =", tunnel.public_url + "/v1")


## Smoke test from the Mac (section 6.3)

Paste the key at the prompt so it stays out of shell history.

```bash
export LLM_URL="https://<your-id>.ngrok-free.app/v1"
read -rs LLM_KEY && export LLM_KEY
curl -s -o /dev/null -w "%{http_code}\n" "$LLM_URL/models"                 # expect 401
curl -s -H "Authorization: Bearer $LLM_KEY" "$LLM_URL/models"               # lists qwen2.5:7b-instruct
curl -s -H "Authorization: Bearer $LLM_KEY" -H "Content-Type: application/json" \
  "$LLM_URL/chat/completions" \
  -d '{"model":"qwen2.5:7b-instruct","messages":[{"role":"user","content":"Reply with OK"}],"max_tokens":5}'
```

Keep this notebook session open. When it stops, the model and the URL go with it.

## Alternative: vLLM (section 6.4)

vLLM checks a Bearer key itself, so ngrok can point straight at it and no proxy is
needed. It installs much more slowly and replaces the notebook's PyTorch. The AWQ
build fits one T4; the T4 has no bfloat16, so use `--dtype half`.
